In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

# 1. 데이터 가져오기

In [2]:
import pandas as pd

In [4]:
import pandas as pd
df = pd.read_csv(r"C:\ai_x\download\sharedata\부동산_250213\최종전국평당분양가격_결측치제외.csv", encoding='cp949')
df.head()

,지역명,연도,월,평당분양가격
0,서울,2013,12,18189.0
1,부산,2013,12,8111.0
2,대구,2013,12,8080.0
3,인천,2013,12,10204.0
4,광주,2013,12,6098.0


# 2. 데이터 전처리_ 지역명의 라벨 인코딩
- 지역명을 라벨인코딩한 지역명2 추가
- 분석할 경우 원핫인코딩까지 할 것을 추천

In [5]:
from sklearn.preprocessing import LabelEncoder
import numpy as np
field_scaler = LabelEncoder()
df_output = df.iloc[:,-1]
df_input = df.iloc[:,:-1]

In [62]:
# 입력변수
field_name = np.array(df['지역명'])
field_name_label = field_scaler.fit_transform(field_name)
df['지역명2'] = field_name_label
df_input_label = pd.concat([df_input,pd.DataFrame(field_name_label)],axis=1)
df_input_label.columns = ['지역명', '연도','월','지역명2']
df_input_label
np_input_label = np.array(df_input_label.iloc[:,1:])
np_input_label
df['지역명'].value_counts()
df['지역명2'].value_counts()

8     128
0     128
2     128
3     128
12    128
13    128
15    128
16    128
9     128
7     128
1     128
10    128
6     128
4     128
11    128
5     128
14    128
Name: 지역명2, dtype: int64

In [7]:
# 타겟변수
np_output = np.array(df_output).reshape(-1,1)
np_output

array([[18189. ],
       [ 8111. ],
       [ 8080. ],
       ...,
       [13827. ],
       [13252.8],
       [25419.9]])

# 3. normalization 스케일 조정
- 입력변수(지역명2, 연도, 월)와 타겟변수(평당분양가격) 따로 스케일 조정(MinMaxScaler)
- 지역명2n, 연도n, 월n 필드 추가

In [67]:
from sklearn.preprocessing import MinMaxScaler
x_scaler_normal = MinMaxScaler()
y_scaler_normal = MinMaxScaler()
scaled_x = x_scaler_normal.fit_transform(np_input_label)
scaled_y = y_scaler_normal.fit_transform(np_output)
# pd.concat(df_input_label,)
pd_scaled_x = pd.DataFrame(scaled_x, columns = ['연도n','월n','지역명2n'])
df_normalization_input = pd.concat([df_input_label,pd_scaled_x], axis=1)
df_normalization_output = pd.DataFrame(scaled_y,columns=['평당분양가격'])
df_normalization_output
# df_normalization_input

,평당분양가격
0,0.328198
1,0.065274
2,0.064466
3,0.119878
4,0.012757
...,...
2171,0.168252
2172,0.195974
2173,0.214398
2174,0.199418


# 4. standardization 스케일 조정
- 입력변수(지역명2, 연도, 월)와 타겟변수(평당분양가격) 따로 스케일 조정(StandardScaler)
- 지역명2s, 연도S, 월s 필드 추가

In [46]:
from sklearn.preprocessing import StandardScaler
x_scaler_stand = StandardScaler()
y_scaler_stand = StandardScaler()
scaled_x = x_scaler_stand.fit_transform(np_input_label)
scaled_y = y_scaler_stand.fit_transform(np_output)
pd_scaled_x = pd.DataFrame(scaled_x,columns=['연도s','월s','지역명2s'])
pd_scaled_y = pd.DataFrame(scaled_y,columns=['평당분양가격'])
df_semi_final_input = pd.concat([df_normalization_input, pd_scaled_x],axis=1)
display(df_semi_final_input), display(pd_scaled_y)

,지역명,연도,월,지역명2,연도n,월n,지역명2n,연도s,월s,지역명2s
0,서울,2013,12,8,0.0,1.000000,0.5000,-1.875367,1.62196,0.000000
1,부산,2013,12,7,0.0,1.000000,0.4375,-1.875367,1.62196,-0.204124
2,대구,2013,12,5,0.0,1.000000,0.3125,-1.875367,1.62196,-0.612372
3,인천,2013,12,11,0.0,1.000000,0.6875,-1.875367,1.62196,0.612372
4,광주,2013,12,4,0.0,1.000000,0.2500,-1.875367,1.62196,-0.816497
...,...,...,...,...,...,...,...,...,...,...
2171,전북,2024,8,13,1.0,0.636364,0.8125,1.664199,0.46374,1.020621
2172,전남,2024,8,12,1.0,0.636364,0.7500,1.664199,0.46374,0.816497
2173,경북,2024,8,3,1.0,0.636364,0.1875,1.664199,0.46374,-1.020621
2174,경남,2024,8,2,1.0,0.636364,0.1250,1.664199,0.46374,-1.224745


,평당분양가격
0,1.168591
1,-0.728312
2,-0.734147
3,-0.334363
4,-1.107203
...,...
2171,0.014639
2172,0.214643
2173,0.347566
2174,0.239489


(None, None)

# 5. 지역명 원핫인코딩

In [11]:
# 원핫인코딩
from tensorflow.keras.utils import to_categorical
onehot_field_label = to_categorical(np_input_label[:,2])
onehot_field_label
df_final = pd.concat([df_semi_final_input,pd.DataFrame(onehot_field_label)],axis=1)
# display(df_final)
# display(pd_scaled_y),display(df_normalization_output)
df_final_output = pd.concat([df_output,pd_scaled_y,df_normalization_output],axis=1)
df_final_output.columns = ['평당분양가격','평당분양가격s','평당분양가격n']
display(df_final_output), display(df_final)

,평당분양가격,평당분양가격s,평당분양가격n
0,18189.0,1.168591,0.328198
1,8111.0,-0.728312,0.065274
2,8080.0,-0.734147,0.064466
3,10204.0,-0.334363,0.119878
4,6098.0,-1.107203,0.012757
...,...,...,...
2171,12058.2,0.014639,0.168252
2172,13120.8,0.214643,0.195974
2173,13827.0,0.347566,0.214398
2174,13252.8,0.239489,0.199418


,지역명,연도,월,지역명2,연도n,월n,지역명2n,연도s,월s,지역명2s,...,7,8,9,10,11,12,13,14,15,16
0,서울,2013,12,8,0.0,1.000000,0.5000,-1.875367,1.62196,0.000000,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,부산,2013,12,7,0.0,1.000000,0.4375,-1.875367,1.62196,-0.204124,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,대구,2013,12,5,0.0,1.000000,0.3125,-1.875367,1.62196,-0.612372,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,인천,2013,12,11,0.0,1.000000,0.6875,-1.875367,1.62196,0.612372,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,광주,2013,12,4,0.0,1.000000,0.2500,-1.875367,1.62196,-0.816497,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2171,전북,2024,8,13,1.0,0.636364,0.8125,1.664199,0.46374,1.020621,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2172,전남,2024,8,12,1.0,0.636364,0.7500,1.664199,0.46374,0.816497,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2173,경북,2024,8,3,1.0,0.636364,0.1875,1.664199,0.46374,-1.020621,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2174,경남,2024,8,2,1.0,0.636364,0.1250,1.664199,0.46374,-1.224745,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


(None, None)

# 정리

In [34]:
# 입력변수 정리
df_origin_input = df_final.iloc[:,:4]
df_origin_input # input 라벨인코딩 결과
df_normal_input = df_final.iloc[:,4:7]
df_normal_input # normalization 입력변수 결과
df_stand_input = df_final.iloc[:,7:10]
df_stand_input # Standardization 입련변수 결과
one_hot = df_final.iloc[:,10:]
one_hot # 지역명 원핫인코딩

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2171,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2172,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2173,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2174,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [69]:
# 타겟변수 정리
df_origin_output = df_final_output.iloc[:,0]
df_stand_output = df_final_output.iloc[:,1]
df_normal_output = df_final_output.iloc[:,2]
df_stand_output

0       1.168591
1      -0.728312
2      -0.734147
3      -0.334363
4      -1.107203
          ...   
2171    0.014639
2172    0.214643
2173    0.347566
2174    0.239489
2175    2.529607
Name: 평당분양가격s, Length: 2176, dtype: float64

In [47]:
# 샘플제작

In [70]:
# 1. normaliztion
# x_scaler_normal, y_scaler_normal
np.column_stack([df_normal_input,df_normal_output])

array([[0.        , 1.        , 0.5       , 0.32819817],
       [0.        , 1.        , 0.4375    , 0.06527439],
       [0.        , 1.        , 0.3125    , 0.06446563],
       ...,
       [1.        , 0.63636364, 0.1875    , 0.21439846],
       [1.        , 0.63636364, 0.125     , 0.19941822],
       [1.        , 0.63636364, 0.875     , 0.51684429]])

In [71]:
# 2. Standardization
# x_scaler_stand, y_scaler_stand
np.column_stack([df_stand_input,df_stand_output])

array([[-1.87536661,  1.62196025,  0.        ,  1.16859132],
       [-1.87536661,  1.62196025, -0.20412415, -0.72831216],
       [-1.87536661,  1.62196025, -0.61237244, -0.73414705],
       ...,
       [ 1.66419932,  0.46374038, -1.02062073,  0.34756602],
       [ 1.66419932,  0.46374038, -1.22474487,  0.23948883],
       [ 1.66419932,  0.46374038,  1.22474487,  2.52960734]])

In [48]:
one_hot

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2171,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2172,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2173,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2174,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# 인코딩한 결과 디코딩하기

In [53]:
# Step 3: 원-핫 → 정수
decoded_int = np.argmax(np.array(one_hot), axis=1)
print("디코딩된 정수:", decoded_int)

# Step 4: 정수 → 문자열 라벨
decoded_labels = field_scaler.inverse_transform(decoded_int)
print("복원된 문자열 라벨:", decoded_labels)  # ['서울', '경기', '부산']


디코딩된 정수: [ 8  7  5 ...  3  2 14]
복원된 문자열 라벨: ['서울' '부산' '대구' ... '경북' '경남' '제주']


In [78]:
loc_info = df[['지역명','지역명2']].head(17).sort_values(by=['지역명2'])
columns = loc_info['지역명'].tolist()

In [81]:
one_hot.columns = columns
one_hot.head(20)

,강원,경기,경남,경북,광주,대구,대전,부산,서울,세종,울산,인천,전남,전북,제주,충남,충북
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
7,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [82]:
pd.options.display.max_columns = 30 # 최대 출력 가능한 데이터 프레임 열수

In [84]:
pd.concat([df_origin,df_normal_input,df_stand_input,one_hot,df_final_output],axis=1)

,지역명,연도,월,지역명2,연도n,월n,지역명2n,연도s,월s,지역명2s,강원,경기,경남,경북,광주,대구,대전,부산,서울,세종,울산,인천,전남,전북,제주,충남,충북,평당분양가격,평당분양가격s,평당분양가격n
0,서울,2013,12,8,0.0,1.000000,0.5000,-1.875367,1.62196,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,18189.0,1.168591,0.328198
1,부산,2013,12,7,0.0,1.000000,0.4375,-1.875367,1.62196,-0.204124,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8111.0,-0.728312,0.065274
2,대구,2013,12,5,0.0,1.000000,0.3125,-1.875367,1.62196,-0.612372,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8080.0,-0.734147,0.064466
3,인천,2013,12,11,0.0,1.000000,0.6875,-1.875367,1.62196,0.612372,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,10204.0,-0.334363,0.119878
4,광주,2013,12,4,0.0,1.000000,0.2500,-1.875367,1.62196,-0.816497,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6098.0,-1.107203,0.012757
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2171,전북,2024,8,13,1.0,0.636364,0.8125,1.664199,0.46374,1.020621,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,12058.2,0.014639,0.168252
2172,전남,2024,8,12,1.0,0.636364,0.7500,1.664199,0.46374,0.816497,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,13120.8,0.214643,0.195974
2173,경북,2024,8,3,1.0,0.636364,0.1875,1.664199,0.46374,-1.020621,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,13827.0,0.347566,0.214398
2174,경남,2024,8,2,1.0,0.636364,0.1250,1.664199,0.46374,-1.224745,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,13252.8,0.239489,0.199418
